# TRIBE v2 — Colab Backend
**Cell 1 installs and auto-restarts. After restart, run Cells 2–8.**

> Recommended: `Runtime → Change runtime type → T4 GPU`

In [ ]:
# CELL 1 — Install everything (auto-restarts at end to flush old numpy)
!pip install -q uv
!uv pip install --system "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"
!uv pip install --system fastapi "uvicorn[standard]" yt-dlp pydantic nest_asyncio huggingface_hub
!apt-get install -y -q ffmpeg 2>/dev/null || true
print("\n✓ Done. Restarting runtime to flush stale numpy from memory...")
import os, time; time.sleep(2)
os.kill(os.getpid(), 9)  # Colab auto-restarts — expected

In [ ]:
# CELL 2 — Verify imports (after restart)
from tribev2.demo_utils import TribeModel
import tribev2, numpy as np, torch
print(f"tribev2 : {tribev2.__file__}")
print(f"numpy   : {np.__version__}")
print(f"torch   : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none — CPU mode'}")
print("TribeModel ✓")

In [ ]:
# CELL 3 — Patch faster_whisper on disk to fix float16 error
#
# Root cause: tribev2 spawns whisperx as a subprocess (separate process).
# Python-level patches don't reach it. The only reliable fix is to patch
# faster_whisper/transcribe.py on disk so every call to WhisperModel
# automatically falls back from float16 → int8 when no GPU supports it.

import inspect, re, torch
import faster_whisper.transcribe as _fwt

src = inspect.getfile(_fwt)
print(f"Patching: {src}")

with open(src) as f:
    code = f.read()

# The error fires at: self.model = ctranslate2.models.Whisper(...)
# Wrap it to retry with int8 if float16 is rejected
old = '            self.model = ctranslate2.models.Whisper('
new = ('            try:\n'
       '                self.model = ctranslate2.models.Whisper(')

if old in code and 'except ValueError as _ct2_err' not in code:
    # Find the full ctranslate2.models.Whisper(...) call (may span multiple lines)
    # and add try/except around it
    pattern = r'(            self\.model = ctranslate2\.models\.Whisper\([^)]+\))'
    match = re.search(pattern, code, re.DOTALL)
    if match:
        original_call = match.group(1)
        indented_call = original_call.replace('            ', '                ')
        replacement = (
            '            try:\n'
            + indented_call + '\n'
            + '            except ValueError as _ct2_err:\n'
            + '                if "float16" in str(_ct2_err):\n'
            + '                    import re as _re\n'
            + '                    _ct2_args = _re.sub(r"compute_type=\\S+", "compute_type=\\"int8\\"", str(' + repr(original_call) + '))\n'
            + '                    self.model = ctranslate2.models.Whisper(\n'
            + '                        model_path, device=device, device_index=device_index,\n'
            + '                        compute_type="int8", inter_threads=inter_threads,\n'
            + '                        intra_threads=intra_threads, files=files)\n'
            + '                else:\n'
            + '                    raise'
        )
        code = code.replace(original_call, replacement)
        with open(src, 'w') as f:
            f.write(code)
        print("faster_whisper patched on disk ✓")
    else:
        print("Pattern not found — trying simpler patch...")
        # Simpler: just replace the compute_type arg in the constructor call
        code2 = re.sub(
            r'(WhisperModel\.__init__[\s\S]*?compute_type\s*=\s*)["\']float16["\']',
            r'\1"int8"',
            code
        )
        if code2 != code:
            with open(src, 'w') as f:
                f.write(code2)
            print("faster_whisper patched (fallback method) ✓")
        else:
            print("Could not patch — will rely on subprocess-level fix")
elif 'except ValueError as _ct2_err' in code:
    print("faster_whisper already patched ✓")
else:
    print("Pattern not found in faster_whisper — may already be fixed")

# Also patch whisperx/asr.py default compute_type from float16 → int8
import whisperx.asr as _wxasr
src2 = inspect.getfile(_wxasr)
with open(src2) as f:
    code2 = f.read()

compute_type = 'float16' if torch.cuda.is_available() else 'int8'
patched2 = re.sub(
    r'(def load_model[\s\S]*?compute_type\s*:\s*str\s*=\s*)["\']float16["\']',
    f'\\1"{compute_type}"',
    code2
)
if patched2 != code2:
    with open(src2, 'w') as f:
        f.write(patched2)
    print(f"whisperx/asr.py default compute_type → {compute_type} ✓")
else:
    print(f"whisperx/asr.py already uses {compute_type} or pattern not found")

In [ ]:
# CELL 4 — Clone / update brain-neuro
import os
repo = '/content/brain-neuro'
if os.path.exists(repo):
    !git -C {repo} pull -q
    print('brain-neuro updated ✓')
else:
    !git clone -q https://github.com/Sambhram1/brain-neuro.git {repo}
    print('brain-neuro cloned ✓')
os.chdir(repo)
print(f'cwd: {os.getcwd()}')

In [ ]:
# CELL 5 — HuggingFace login
import huggingface_hub
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('HF_TOKEN from Colab secrets ✓')
except Exception:
    hf_token = input('Enter HuggingFace token (huggingface.co/settings/tokens): ')
huggingface_hub.login(token=hf_token, add_to_git_credential=False)
print('Logged in ✓  — weights download on first /analyze call')

In [ ]:
# CELL 6 (Optional) — cookies.txt for Instagram/TikTok. Skip for YouTube.
import shutil, os
try:
    from google.colab import files
    print('Upload cookies.txt or skip this cell')
    uploaded = files.upload()
    for fname in uploaded:
        shutil.move(fname, 'cookies.txt')
        print('cookies.txt ✓')
except ImportError:
    print('cookies.txt ✓' if os.path.exists('cookies.txt') else 'No cookies.txt — YouTube only')

In [ ]:
# CELL 7 — Cloudflared tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('cloudflared ✓')

In [ ]:
# CELL 8 — Start backend + tunnel
import nest_asyncio, uvicorn, threading, subprocess, time, re, os
os.chdir('/content/brain-neuro')
nest_asyncio.apply()

def run_server():
    uvicorn.run('backend:app', host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=run_server, daemon=True).start()
time.sleep(4)
print('FastAPI running on :8000 ✓')

proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print('Waiting for tunnel...')
for line in proc.stdout:
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line.decode())
    if match:
        url = match.group()
        print(f'\n{"═"*54}')
        print(f'  BACKEND URL: {url}')
        print(f'{"═"*54}')
        print('→ Paste into the frontend Backend field')
        break